In [0]:
%run  /Shared/iplDatabricks/spn_adf_databricks_conn

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
silver_path = "abfss://silver@ipldalalakestorage.dfs.core.windows.net/"
gold_path = "abfss://gold@ipldalalakestorage.dfs.core.windows.net/"

In [0]:

def write_gold_table(df, table_name: str):

    df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("mergeSchema", "true") \
        .save(f"{gold_path}{table_name}")

In [0]:
silver_df = spark.read.format("delta").load(silver_path)

In [0]:
display(silver_df)

In [0]:
batsman_wayof_out = (
    silver_df.select('match_id','striker','bowler','wicket_type')
    .filter(col('wicket_type').isNotNull())
)
display(batsman_wayof_out)
write_gold_table(batsman_wayof_out, "batsman_wayof_out")

In [0]:
dismissal_types = [
    "stumped", "hit wicket", "bowled", "lbw", 
    "caught and bowled", "retired hurt", "caught", "run out"
]

batsman_outWithNoOfWicketTypes = (
    batsman_wayof_out.filter(col("wicket_type").isin(dismissal_types))
    .groupBy("striker")
    .pivot("wicket_type", dismissal_types)
    .count()
    .na.fill(0)
    .withColumnRenamed("striker", "batsman")
)

batsman_outWithNoOfWicketTypes = batsman_outWithNoOfWicketTypes.toDF(
    *[c.replace(" ", "_") for c in batsman_outWithNoOfWicketTypes.columns]
)

# display(batsman_outWithNoOfWicketTypes) 
write_gold_table(batsman_outWithNoOfWicketTypes, "batsman_outWithNoOfWicketTypes")

In [0]:
series_bowler_perf = (
    silver_df.groupBy("bowler")
    .agg(
        sum("total_runs").alias("total_runs"),
        sum(
            when(
                (coalesce(col("wides"), lit(0)) == 0) & (coalesce(col("noballs"), lit(0)) == 0),
                1
            ).otherwise(0)
        ).alias("legal_balls"),
        sum(
            when(
                col("wicket_type").isNotNull() & ~col("wicket_type").isin("run out", "retired hurt", "obstructing the field"),
                1
            ).otherwise(0)
        ).alias("total_wickets")
    )
    # Calculate total overs for the series
    .withColumn("complete_overs", (col("legal_balls") / 6).cast(IntegerType()))
    .withColumn("rem_balls", col("legal_balls") % 6)
    .withColumn("total_overs", concat_ws(".", col("complete_overs"), col("rem_balls")))
    
    # Economy calculation across the entire series
    .withColumn("economy", round((col("total_runs") / col("legal_balls")) * 6, 2))
    .drop("legal_balls", "complete_overs", "rem_balls")
)

# display(series_bowler_perf)
write_gold_table(series_bowler_perf, "series_bowler_perf")

In [0]:
# Updated rule checks using direct function calls
is_legal_delivery = ((coalesce(col("wides"), lit(0)) == 0) & (coalesce(col("noballs"), lit(0)) == 0))
is_bowler_wicket = col("wicket_type").isNotNull() & (~col("wicket_type").isin("run out", "retired hurt", "obstructing the field"))

match_bowler_perf = (
    silver_df.groupBy("match_id", "bowler")
    .agg(
        # Total runs conceded by the bowler on their deliveries
        sum("total_runs").alias("total_runs"),
        
        # Count legal balls to compute exact overs
        sum(when(is_legal_delivery, 1).otherwise(0)).alias("legal_balls"),
        
        # Count valid bowler wickets
        sum(when(is_bowler_wicket, 1).otherwise(0)).alias("total_wickets")
    )
    # Calculate total overs (Complete Overs + Remaining Balls as decimals, e.g., 3.2 overs)
    .withColumn("complete_overs", (col("legal_balls") / 6).cast(IntegerType()))
    .withColumn("rem_balls", col("legal_balls") % 6)
    .withColumn("total_overs", concat_ws(".", col("complete_overs"), col("rem_balls")))
    
    # Economy = (Total Runs Conceded / Total Legal Balls) * 6
    .withColumn("economy", round(try_divide(col("total_runs"), col("legal_balls")) * 6, 2))
    .drop("legal_balls", "complete_overs", "rem_balls")
    .orderBy('match_id', 'bowler')
)

# display(match_bowler_perf)
write_gold_table(match_bowler_perf, "match_bowler_perf")


In [0]:
# Aggregate runs scored by batsman against bowler archetypes
gold_batsman_vs_bowler_type = (
    silver_df.groupBy("striker", "bowler")
    .agg(sum('runs_off_bat').alias('runs'))
    .orderBy('striker')
    
)
# display(gold_batsman_vs_bowler_type)
write_gold_table(gold_batsman_vs_bowler_type, "batsman_vs_bowler")

In [0]:
# Aggregate runs scored by batsman
gold_batsman_runs = (
    silver_df.groupBy("striker")
    .agg(sum('runs_off_bat').alias("batsman's_runs"))
    .orderBy('striker')
    
)
# display(gold_batsman_runs)
write_gold_table(gold_batsman_runs, "batsman's_runs")